---
technologies: "Celery"
category: "Choice and use of technology"
difficulty: "Intermediate"
---

# Celery

## Used material

1. <span id="used-material-1"></span> [Run Redis on Docker](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/docker/)

2. <span id="used-material-2"></span> [Redis pip package](https://pypi.org/project/redis/)

3. <span id="used-material-3"></span> [Celery pip package](https://pypi.org/project/celery/)
   
4. <span id="used-material-4"></span> [Celery user guide](https://docs.celeryq.dev/en/latest/userguide/index.html)

5. <span id="used-material-5"></span> [Celery first steps](https://docs.celeryq.dev/en/stable/getting-started/first-steps-with-celery.html)

## Why use Celery?

Celery was chosen for the following reasons:

- One of the most used distributed task queues for Python (mature)

- Enables easy creation and execution of Python-based automation code (abstracted)

- Usable in any place where Python can be used (interoperable)

These enable us to use Celery to automate small, repetitive interactions across separate software, containers, and systems without changing development methods.

## How to use Celery?

Assuming we have a running Redis container [(1)](#used-material-1), we can start the Celery backend by setting up a venv |[(2)](#used-material-2),[(3)](#used-material-3)| and executing the [run](./celery/run_celery.py) file in the following way |[(4)](#used-material-4), [(5)](#used-material-5)|:

```
cd celery
python3 -m venv celery_venv
source celery_venv/bin/activate
pip install -r packages.txt
python3 run_celery.py
```

We can now run the interactive [submitter-requests](./celery/tasks/interactive/requests.py) task with input parameters by sending task requests with the abstracted Celery task interaction functions [setup](./celery_func/setup.py) and [use](./celery_func/use.py) that send the task and get the result in the following way:

In [1]:
%run ./celery_func/setup.py
%run ./celery_func/use.py

In [2]:
redis_endpoint = '127.0.0.1'
redis_port = '6379'
redis_db = '0'

parameters_dict = {
    'celery-cache-location': 'local',
    'celery-lock-location': 'local',
    'cache': {
        'local': {
            'system': 'redis',
            'user': 'part-2',
            'target': 'backend',
            'redis-endpoint': redis_endpoint,
            'redis-port': redis_port,
            'redis-db': redis_db,
        }
    },
    'lock': {
        'local': {
            'system': 'redis',
            'user': 'part-2',
            'target': 'backend',
            'redis-endpoint': redis_endpoint,
            'redis-port': redis_port,
            'redis-db': redis_db,
        }
    }
}

In [3]:
celery_client = celery_setup_instance(
    redis_endpoint = redis_endpoint,
    redis_port = redis_port,
    redis_db = redis_db 
)

In [4]:
request_task_id = celery_get_id(
    task_name = 'tasks.submitter-requests',
    task_kwargs = {
        'mode': 'logs',
        'input': parameters_dict 
    }
)
print(request_task_id)

bd6195df-df9e-48b7-9445-c0170ab86f42


In [5]:
interactive_task_data = celery_get_task( 
    celery_client = celery_client,
    task_id = request_task_id
)

In [6]:
interactive_task_data

{'status': 'SUCCESS',
 'result': {'logs': ['[2026-03-25 11:00:03,477: INFO/MainProcess] Connected to redis://127.0.0.1:6379/0',
   '[2026-03-25 11:00:03,482: INFO/MainProcess] mingle: searching for neighbors',
   '[2026-03-25 11:00:04,497: INFO/MainProcess] mingle: all alone',
   '[2026-03-25 11:00:04,528: INFO/MainProcess] celery@ submitter_backend ready.',
   '[2026-03-25 11:00:13,441: INFO/MainProcess] Task tasks.submitter-requests[bd6195df-df9e-48b7-9445-c0170ab86f42] received',
   '[2026-03-25 11:00:13,444: WARNING/ForkPoolWorker-7] Testing connection: 127.0.0.1:6379:0',
   '[2026-03-25 11:00:13,448: WARNING/ForkPoolWorker-7] redis connected: True',
   '[2026-03-25 11:00:13,449: WARNING/ForkPoolWorker-7] Checking lock for lock:part-2:backend:interactive:submitter-requests',
   '[2026-03-25 11:00:13,449: WARNING/ForkPoolWorker-7] Lock active: False',
   '[2026-03-25 11:00:13,449: WARNING/ForkPoolWorker-7] Lock lock:part-2:backend:interactive:submitter-requests active: False',
   '[

Now, to run the scheduled [submitter-trigger](./celery/tasks/scheduled/trigger.py) task without parameters, we need to use [caching](./caching.py) functions to give the necessary parameters. We can wait for the task to complete in the following way:

In [7]:
%run ./caching.py

In [8]:
cache_location = parameters_dict['celery-cache-location']
cache_parameters = parameters_dict['cache'][cache_location]

parameters_key = caching_generate_key(
    static = True,
    user = cache_parameters['user'],
    target = cache_parameters['target'],
    group = 'dict'
)

print(parameters_key)

cache:part-2:backend:dict


In [9]:
cache_response = caching_save_dict(
    cache_parameters = cache_parameters,
    cache_key = parameters_key,
    nested_dict = parameters_dict
)

Testing connection: 127.0.0.1:6379:0
Using key: cache:part-2:backend:dict
redis responded with: True


In [10]:
scheduled_task_data = celery_await_signature(
    celery_client = celery_client,
    task_name = 'tasks.submitter-trigger',
    task_kwargs = {},
    timeout = 20
)

In [11]:
scheduled_task_data

{'status': 'SUCCESS', 'result': {'key': 'value'}}

## Celery file structure

When you check the files inside the celery folder, you will notice the following structure:

- [packages](./celery/packages.txt)
- [run_celery](./celery/run_celery.py)
- [setup_celery](./celery/setup_celery.py)
- logs:
    - [backend-logs](./celery/logs/backend-logs.txt)
- functions:
    - [caching](./celery/functions/caching.py)
    - [concurrency](./celery/functions/concurrency.py)
    - platforms:
        - celery:
            - [setup](./celery/functions/platforms/celery/setup.py)
            - [use](./celery/functions/platforms/celery/use.py) 
        - redis:
            - [setup](./celery/functions/platforms/redis/setup.py)
            - [use](./celery/functions/platforms/redis/use.py)
- tasks:
    -  interactive
        - [requests](./celery/tasks/interactive/requests.py)
    -  scheduled
        - [trigger](./celery/tasks/scheduled/trigger.py)

The purpose is again easier understandability and development that enables us to clearly divide the responsibility in return for more complex file structures.

## Important parts of Celery

The most important parts to keep an eye on when trying to understand or develop a Celery application are:

- Module = A named instance with a context (see use in [setup](./celery/functions/platforms/celery/setup.py) rows 8-15)
- Tasks = Executable functions assigned to module (see use in [requests](./celery/tasks/interactive/requests.py) rows 8-25)
- Signatures = The execution name for a task (see use in [requests](./celery/tasks/interactive/requests.py) row 20)

---